In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Somali-tts/somali-tts-datasets", 
    repo_type="dataset", local_dir="./somali-tts-datasets", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 1 files: 100%|██████████| 1/1 [00:02<00:00,  2.91s/it]


'/home/ubuntu/somali-tts-datasets'

In [3]:
files = glob('somali-tts-datasets/*/*.parquet')
len(files)

1

In [4]:
df = pd.read_parquet(files[0])
df.head()

,text,audio,__index_level_0__
0,Aamina naftaada wax walba waad awoodaa inaa sa...,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,1
1,Aaminaada naftaada waa mid kaa caawinaysa inaa...,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,2
2,Afafka qalaad barashadoodu waxay furaan fursad...,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,3
3,Akhrinta sheekooyinka carruurta waxay horumari...,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,4
4,Akhrisku waa furaha guusha waxbarasho,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,5


In [7]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            try:
                t = df['text'].iloc[i].strip()
                if len(t) < 2:
                    continue
                audio_filename = f'{f_new}_{i}.mp3'
                audio_filename = os.path.join(base, audio_filename)
                b = df['audio'].iloc[i]['bytes']
                audio_np, sr = sf.read(io.BytesIO(b))
                if audio_np.ndim > 1:
                    audio_np = audio_np.mean(axis=1)
                if audio_np.shape[0] < 10000:
                    continue
                sf.write(audio_filename, audio_np, sr)
                
                data.append({
                    'audio_filename': audio_filename,
                    'text': t,
                    'speaker': f"{base}"
                })
            except:
                pass
        
    return data

In [9]:
data = loop((files[:1], 0))

In [10]:
with open('somali-tts-datasets.json', 'w') as fopen:
    json.dump(data, fopen)

In [11]:
audio_files = [d['audio_filename'] for d in data]

with open('somali-tts-datasets-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [12]:
len(data)

1719

In [15]:
# !zip -rq somali-tts-datasets_audio.zip somali-tts-datasets_audio

In [16]:
# !hf upload malaysia-ai/Multilingual-TTS somali-tts-datasets_audio.zip --repo-type=dataset